# 13 — 3D element maps

This notebook focuses on inspecting reconstructed 3D ion volumes.

It assumes notebook 12 has produced:

```text
data/bins/channel_bins_from_3d_csv_calibration.csv
```

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import re

import pandas as pd
import plotly.io as pio

from pymagsims import SIMSVolume
from pymagsims.plotting import (
    plot_ion_image,
    plot_array_slider,
    plot_array_grid_slider,
)

pio.renderers.default = "iframe"

DATA = Path("../data")
RAW_LAYER_DIR = DATA / "3d"
IMAGE_SHAPE = (256, 256)

CHANNEL_BIN_FILE = DATA / "bins" / "channel_bins_from_3d_csv_calibration.csv"

## 1. Load paths and bins

In [ ]:
def natural_sort_key(path):
    return [
        int(text) if text.isdigit() else text.lower()
        for text in re.split(r"(\d+)", path.name)
    ]

paths = sorted(RAW_LAYER_DIR.glob("*Image_*.raw"), key=natural_sort_key)
selected_bins = pd.read_csv(CHANNEL_BIN_FILE)

print(f"Found {len(paths)} raw image layers")
display(selected_bins.head())
print("Available elements:", sorted(selected_bins["element"].dropna().unique()))

## 2. Reconstruct individual bin volumes

In [ ]:
volume = SIMSVolume.from_fpd_raw_image_series(
    paths=paths,
    bins=selected_bins,
    spectrum=None,
    include_total=True,
    shape=IMAGE_SHAPE,
)

print("Number of volume labels:", len(volume.labels()))
print(volume.labels()[:20])

## 3. Sum reconstructed bins by element

This happens **after** reconstruction to avoid merging channel ranges that may contain unrelated peaks.

In [ ]:
# Choose elements present in your selected bins.
elements = ["Si", "Ti", "Mg", "Hf"]

element_volumes = {}

for element in elements:
    try:
        element_volumes[element] = volume.summed_by_element(selected_bins, element)
        print(element, element_volumes[element].shape, element_volumes[element].sum())
    except ValueError as exc:
        print(f"Skipping {element}: {exc}")

element_volumes

## 4. Plot one element with a layer slider

In [ ]:
# Example: plot the first available element.
first_element = next(iter(element_volumes))

plot_array_slider(
    element_volumes[first_element],
    label=first_element,
    log=True,
    colorscale="Magma",
).show()

## 5. Plot summed projections

In [ ]:
for element, arr in element_volumes.items():
    plot_ion_image(
        arr.sum(axis=0),
        log=True,
        title=f"{element} summed projection",
    )

## 6. 2x2 / 3x3 grid with one shared layer slider

In [ ]:
arrays = {
    "Total": volume.get("Total"),
    **element_volumes,
}

plot_array_grid_slider(
    arrays,
    log=True,
    nrows=2,
    ncols=3,
    colorscale={
        "Total": "Gray",
        "Si": "Blues",
        "Ti": "Magma",
        "Mg": "Plasma",
        "Hf": "Cividis",
        "Au": "Inferno",
        "Pt": "Viridis",
        "W": "Turbo",
        "Cr": "Greens",
    },
    width=1100,
    height=850,
).show()

## Notes

If an element is missing, check:

```python
selected_bins[selected_bins["element"] == "Au"]
volume.labels()
```

In [ ]:
selected_bins[selected_bins["element"] == "Mg"]
volume.labels()